# 03 — Data Preparation

This notebook prepares NASA C-MAPSS FD001 for RUL modeling. In Colab, select **Runtime → Run all** and upload `train_FD001.txt` when prompted. In local Jupyter, set `INPUT_PATH` to the file location. Dependencies: pandas and NumPy. [NASA dataset and download](https://zenodo.org/records/15346912).

The official test trajectories and labels are reserved for final evaluation.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json

import numpy as np
import pandas as pd

INPUT_PATH = Path("train_FD001.txt")
SPLIT_SEED = 42
TRAIN_ENGINE_COUNT = 80

if INPUT_PATH.is_file():
    raw_bytes = INPUT_PATH.read_bytes()
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Set INPUT_PATH to train_FD001.txt: {INPUT_PATH}") from exc
    uploaded = files.upload()
    if len(uploaded) != 1 or "train_FD001.txt" not in uploaded:
        raise ValueError("Upload only train_FD001.txt, extracted from the dataset ZIP.")
    raw_bytes = uploaded["train_FD001.txt"]

settings = [f"setting_{i}" for i in range(1, 4)]
sensors = [f"sensor_{i}" for i in range(1, 22)]
schema = ["unit", "cycle", *settings, *sensors]

raw = pd.read_csv(BytesIO(raw_bytes), sep=r"\s+", header=None, index_col=False,
                  dtype="float64", on_bad_lines="error")
if raw.shape != (20_631, 26):
    raise ValueError(f"Expected FD001 training shape (20631, 26), got {raw.shape}.")
raw.columns = schema

if not np.isfinite(raw.to_numpy()).all():
    raise ValueError("Missing or non-finite measurements found.")
keys = raw[["unit", "cycle"]].to_numpy()
if (keys <= 0).any() or (keys != np.floor(keys)).any():
    raise ValueError("Engine IDs and cycles must be positive integers.")
raw = raw.astype({"unit": "int64", "cycle": "int64"})
if raw["unit"].nunique() != 100 or raw.duplicated(["unit", "cycle"]).any():
    raise ValueError("Expected 100 engines and unique engine-cycle pairs.")
groups = raw.groupby("unit", sort=True)
if not groups["cycle"].first().eq(1).all() or not groups["cycle"].diff().dropna().eq(1).all():
    raise ValueError("Each engine must start at cycle 1 and advance one cycle at a time.")

print(f"Quality checks passed: {len(raw):,} rows, {raw.shape[1]} columns, 100 engines.")
display(raw.head())

Saving train_FD001.txt to train_FD001.txt
Quality checks passed: 20,631 rows, 26 columns, 100 engines.


,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393.0,2388.0,100.0,38.90,23.4044


## RUL target and engine split

RUL is the final recorded cycle minus the current cycle for each engine. The target uses the complete trajectory to label past observations. The final cycle is used only in this calculation. The 80/20 split is by engine, so every cycle from the same engine stays together.

In [2]:
data = raw.sort_values(["unit", "cycle"]).reset_index(drop=True)
final_cycles = data.groupby("unit")["cycle"].transform("max")
data["RUL"] = final_cycles - data["cycle"]

last_rows = data.groupby("unit").tail(1)
if not data["RUL"].ge(0).all() or not last_rows["RUL"].eq(0).all():
    raise ValueError("RUL must be nonnegative and zero on each final cycle.")
steps = data.groupby("unit")["RUL"].diff().dropna()
if not steps.eq(-1).all():
    raise ValueError("RUL must decrease by one per observed cycle.")

engine_ids = np.sort(data["unit"].unique())
shuffled_ids = np.random.default_rng(SPLIT_SEED).permutation(engine_ids)
train_ids = sorted(shuffled_ids[:TRAIN_ENGINE_COUNT].tolist())
validation_ids = sorted(shuffled_ids[TRAIN_ENGINE_COUNT:].tolist())

train = data[data["unit"].isin(train_ids)].copy()
validation = data[data["unit"].isin(validation_ids)].copy()
if (len(train_ids), len(validation_ids)) != (80, 20):
    raise ValueError("Expected an 80/20 split by engine.")
if set(train["unit"]) & set(validation["unit"]) or len(train) + len(validation) != len(data):
    raise ValueError("Engine groups overlap or fail to cover the dataset.")

split_summary = pd.DataFrame([
    {"split": "train", "engines": len(train_ids), "rows": len(train)},
    {"split": "validation", "engines": len(validation_ids), "rows": len(validation)},
])
display(split_summary)

,split,engines,rows
0,train,80,16527
1,validation,20,4104


## Feature decisions

Check constant settings and sensors using the training engines only. The low-variability flags use the same exploratory thresholds as Data Understanding and remain diagnostic. The baseline uses current measurements as features. Current cycle and engine ID are kept as tracking columns. Scaling can be fitted in the model pipeline using its training data. No RUL cap or rolling features are applied in this baseline.

In [3]:
candidates = [*settings, *sensors]
values = train[candidates]
minimum = values.min()
maximum = values.max()
scale = pd.concat([minimum.abs(), maximum.abs()], axis=1).max(axis=1)
audit = pd.DataFrame({
    "unique_values": values.nunique(),
    "relative_range": (maximum - minimum).div(scale.replace(0, np.nan)).fillna(0),
    "dominant_share": [values[c].value_counts(normalize=True).iloc[0] for c in candidates],
}, index=candidates)
audit.index.name = "column"
audit["constant"] = audit["unique_values"].eq(1)
audit["low_variability_candidate"] = ~audit["constant"] & (
    audit["relative_range"].le(0.001) | audit["dominant_share"].ge(0.99)
)
audit["decision"] = np.where(audit["constant"], "remove", "retain")
audit["reason"] = np.where(audit["constant"], "Constant across training rows",
                           "Available at prediction time and varies in training")
features = audit.index[~audit["constant"]].tolist()
if not features or set(features) & {"unit", "cycle", "RUL", "final_cycle"}:
    raise ValueError("Invalid model feature list.")

correlation = values[features].corr()
upper = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(upper).stack().rename("pearson_r").reset_index()
pairs.columns = ["feature_a", "feature_b", "pearson_r"]
strong_pairs = pairs.loc[pairs["pearson_r"].abs().ge(0.9)].copy()

display(audit)
display(strong_pairs)
print("Model input columns:", features)

,unique_values,relative_range,dominant_share,constant,low_variability_candidate,decision,reason
column,,,,,,,
setting_1,155,2.000000,0.020572,False,False,retain,Available at prediction time and varies in tra...
setting_2,13,2.000000,0.102196,False,False,retain,Available at prediction time and varies in tra...
setting_3,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_1,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_2,309,0.005151,0.009197,False,False,retain,Available at prediction time and varies in tra...
sensor_3,2902,0.028369,0.001513,False,False,retain,Available at prediction time and varies in tra...
sensor_4,3870,0.039057,0.000968,False,False,retain,Available at prediction time and varies in tra...
sensor_5,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_6,2,0.000463,0.980698,False,True,retain,Available at prediction time and varies in tra...


,feature_a,feature_b,pearson_r
103,sensor_9,sensor_14,0.963886


Model input columns: ['setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


## Export and handoff

The two table files keep tracking columns beside the approved features and target. Select model inputs using the ordered list in `feature_spec.json`. For Isolation Forest, select the same approved feature list and omit the supervised target.

In [4]:
output_dir = Path("data_preparation_outputs")
output_dir.mkdir(exist_ok=True)
tracking = ["unit", "cycle"]
export_columns = [*tracking, *features, "RUL"]

train[export_columns].to_csv(output_dir / "train.csv", index=False)
validation[export_columns].to_csv(output_dir / "validation.csv", index=False)
pd.DataFrame(
    [(u, "train") for u in train_ids] + [(u, "validation") for u in validation_ids],
    columns=["unit", "split"],
).to_csv(output_dir / "engine_split.csv", index=False)
audit.to_csv(output_dir / "feature_audit.csv")
strong_pairs.to_csv(output_dir / "strong_correlations.csv", index=False)
split_summary.to_csv(output_dir / "split_summary.csv", index=False)

spec = {
    "source_file": "train_FD001.txt",
    "source_sha256": hashlib.sha256(raw_bytes).hexdigest(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "split_seed": SPLIT_SEED,
    "engine_split_ratio": "80/20",
    "feature_columns": features,
    "removed_constant_columns": audit.index[audit["constant"]].tolist(),
    "tracking_columns": tracking,
    "target_column": "RUL",
    "target_definition": "Final recorded cycle minus current cycle, within engine",
    "scaling": "not applied",
    "rolling_features": "not applied",
}
(output_dir / "feature_spec.json").write_text(json.dumps(spec, indent=2) + "\n", encoding="utf-8")

handoff = """# FD001 Data Preparation Handoff

Extract the ZIP. Run the following from the directory containing the exported files.

```python
import json
from pathlib import Path

import pandas as pd

spec = json.loads(Path("feature_spec.json").read_text(encoding="utf-8"))
features = spec["feature_columns"]
target = spec["target_column"]
train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")

X_train, y_train = train[features], train[target]
X_validation, y_validation = validation[features], validation[target]
```

Use the same engine split for model comparison. Use unit as the group key if
running grouped cross-validation within the training set. Fit any scaler in
the model pipeline on training rows only, or on the training fold in each CV split.
For Isolation Forest, use the listed feature columns without the RUL target.
For final evaluation, select the same feature columns from the official test data
and apply the chosen model's fitted preprocessing. Compare predictions at each
test engine's last observed cycle with its official RUL label.
"""
(output_dir / "README.md").write_text(handoff, encoding="utf-8")

archive = Path("data_preparation_results.zip")
members = [
    "train.csv", "validation.csv", "engine_split.csv", "feature_audit.csv",
    "strong_correlations.csv", "split_summary.csv", "feature_spec.json", "README.md",
]
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in members:
        bundle.write(output_dir / name, arcname=name)

print(f"Created {archive}. In Colab, download it from the Files panel.")

Created data_preparation_results.zip. In Colab, download it from the Files panel.
